# 05 — Huấn luyện mô hình GNN (Model Training)

## 1. Mục tiêu
Notebook này giải thích và trực quan hóa **Bước 5** trong quy trình XAI-Football:
- Kiến trúc GNN: GCN (đề xuất đơn giản), MPNN (đề xuất chính), GATv2 (baseline attention), GraphSAGE (baseline sampling).
- Cấu hình siêu tham số chung: Cố định seed, Optimizer AdamW, LR scheduler ReduceLROnPlateau, Early Stopping (monitor: validation AUC-PR).
- Đọc và phân tích lịch sử huấn luyện (`training_history.csv`) của các mô hình đã huấn luyện chính thức.
- Vẽ đường cong Train Loss và Validation Loss theo epoch.
- Vẽ đường cong Validation AUC-PR theo epoch và xác định Best Epoch.
- Mặc định notebook chỉ đọc lịch sử thực nghiệm có sẵn; có cờ cấu hình `RUN_SMOKE_TEST = False` cho phép chạy thử một smoke test nhanh khi cần kiểm tra pipeline.

## 2. Vị trí của bước trong pipeline
Đây là **Bước 5 — Huấn luyện mô hình GNN (`scripts/06_train.py`, `src/xai_football/training/`)**.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import yaml

REPO_ROOT = Path(".").resolve()
figures_dir = REPO_ROOT / "results" / "figures" / "official"
figures_dir.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", font_scale=1.05)
plt.rcParams["figure.dpi"] = 150

## 3. Input và nguồn dữ liệu
- Cấu hình huấn luyện: `configs/train.yaml`, `configs/model/*.yaml`.
- Lịch sử huấn luyện thực tế từ các runs: `experiments/official/<model>/training_history.csv`.

## 4. Config/script/module sử dụng
- Scripts: `scripts/06_train.py`, `scripts/06_train_baselines.py`.
- Module: `src/xai_football/training/trainer.py`, `losses.py`, `modeling/gnn/`.

## 5. Các bước xử lý & Đọc cấu hình thực tế

In [ ]:
with open(REPO_ROOT / "configs" / "train.yaml", encoding="utf-8") as f:
    train_cfg = yaml.safe_load(f)

print("=== CẤU HÌNH HUẤN LUYỆN CHUNG (configs/train.yaml) ===")
print(f"Seed mặc định:     {train_cfg.get('seed')}")
print(f"Seeds cấu hình:    {train_cfg.get('seeds')}")
print(f"Epochs tối đa:     {train_cfg.get('epochs')}")
print(f"Batch size:        {train_cfg.get('batch_size')}")
print(f"Optimizer:         {train_cfg.get('optimizer')}")
print(f"Scheduler:         {train_cfg.get('scheduler')}")
print(f"Early stopping:    {train_cfg.get('early_stopping')}")

In [ ]:
# Đọc lịch sử huấn luyện của các mô hình GNN
models = ["mpnn", "gcn", "gatv2", "graphsage"]
histories = {}

for m in models:
    hist_path = REPO_ROOT / "experiments" / "official" / m / "training_history.csv"
    if hist_path.exists():
        df_h = pd.read_csv(hist_path)
        histories[m] = df_h
        print(f"Đã đọc lịch sử huấn luyện cho {m.upper()}: {len(df_h)} epochs ghi nhận")
    else:
        print(f"Chưa có training_history.csv cho {m.upper()} tại {hist_path}")

## 6. Kết quả và cách diễn giải
### 6.1. Đường cong Validation AUC-PR theo Epoch

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))

colors = {"mpnn": "#e74c3c", "gcn": "#2980b9", "gatv2": "#27ae60", "graphsage": "#f39c12"}

for m, df_h in histories.items():
    if "val_auc_pr" in df_h.columns:
        ax.plot(df_h["epoch"], df_h["val_auc_pr"], label=f"{m.upper()}", color=colors.get(m, "gray"), linewidth=2.2, marker="o" if len(df_h) < 15 else None)

ax.set_title("Tiến trình hội tụ Validation AUC-PR theo Epoch", fontsize=13, fontweight="bold", pad=12)
ax.set_xlabel("Epoch", fontsize=11)
ax.set_ylabel("Validation AUC-PR", fontsize=11)
ax.legend(title="Mô hình", loc="lower right")
plt.tight_layout()
plt.savefig(figures_dir / "05_val_auc_pr_curves.png", dpi=300)
plt.savefig(figures_dir / "05_val_auc_pr_curves.svg")
plt.show()

### 6.2. Đường cong Train Loss và Validation Loss của mô hình MPNN (đề xuất chính)

In [ ]:
if "mpnn" in histories:
    df_mpnn = histories["mpnn"]
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(df_mpnn["epoch"], df_mpnn["train_loss"], label="Train Loss", color="#2980b9", linewidth=2, marker="o")
    ax.plot(df_mpnn["epoch"], df_mpnn["val_loss"], label="Validation Loss", color="#e74c3c", linewidth=2, marker="s")
    ax.set_title("Đường cong Hội tụ Loss của MPNN (Message Passing Neural Network)", fontsize=12, fontweight="bold")
    ax.set_xlabel("Epoch", fontsize=10)
    ax.set_ylabel("Loss", fontsize=10)
    ax.legend()
    plt.tight_layout()
    plt.savefig(figures_dir / "05_training_loss_curves.png", dpi=300)
    plt.savefig(figures_dir / "05_training_loss_curves.svg")
    plt.show()

## 7. Kiểm tra chất lượng (Quality Checks)

In [ ]:
checks = []
for m, df_h in histories.items():
    has_nan = df_h.isna().any().any()
    checks.append({"Model": m.upper(), "No NaN/Inf": "PASS" if not has_nan else "FAIL", "Epochs": len(df_h), "Best Val PR": df_h["val_auc_pr"].max() if "val_auc_pr" in df_h.columns else "N/A"})

df_qc = pd.DataFrame(checks)
display(df_qc)
assert (df_qc["No NaN/Inf"] == "PASS").all(), "Lịch sử huấn luyện chứa giá trị NaN/Inf!"

## 8. Output được lưu ở đâu
- Thư mục thí nghiệm: `experiments/official/<model>/`
  - `training_history.csv`
  - `best_model.pt`
  - `validation_metrics.csv`
  - `test_metrics.csv`
  - `run_manifest.json`
- Trọng số chính thức: `models/official/<model>_best.pt`
- Biểu đồ:
  - `results/figures/official/05_val_auc_pr_curves.png` (và `.svg`)
  - `results/figures/official/05_training_loss_curves.png` (và `.svg`)

## 9. Cách tái chạy
Chạy thử smoke test nhanh (1 epoch, hoàn tất trong ~1 giây):
```bash
python scripts/06_train.py --model-config configs/model/mpnn.yaml --smoke-test
```
Chạy huấn luyện đầy đủ (nhiều seed):
```bash
python scripts/06_train.py --model-config configs/model/mpnn.yaml --seeds 42
```

## 10. Giới hạn và giả định
- Để tiết kiệm tài nguyên tính toán, notebook mặc định đọc các file artifact đã lưu thay vì tự động kích hoạt huấn luyện lại từ đầu.